In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, Dropout

In [ ]:
df = pd.read_csv('/content/Restaurant_Reviews.tsv', sep='\t')
print(df.head())
print(df.shape)

                                              Review  Liked
0                           Wow... Loved this place.      1
1                                 Crust is not good.      0
2          Not tasty and the texture was just nasty.      0
3  Stopped by during the late May bank holiday of...      1
4  The selection on the menu was great and so wer...      1
(1000, 2)


In [ ]:
df = df[['Review','Liked']].dropna()
df['Review'] = df['Review'].astype(str).str.strip()
df['Liked'] = df['Liked'].astype(int)
print(df['Liked'].value_counts())

Liked
1    500
0    500
Name: count, dtype: int64


In [ ]:
X = df['Review'].values
y = df['Liked'].values

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
print("Training samples:",len(X_train))
print("Testing samples:",len(X_test))

Training samples: 800
Testing samples: 200


In [ ]:
max_words = 10000
max_len = 100
tokenizer = Tokenizer(num_words=max_words,oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)
X_train_pad = pad_sequences(X_train_seq,maxlen=max_len,padding='post',truncating='post')
X_test_pad = pad_sequences(X_test_seq,maxlen=max_len,padding='post',truncating='post')
print(X_train_pad.shape)

(800, 100)


In [ ]:
"""
The food was good
Tokenization
[the, food, was, good]
Text_to_sequence
[1,2,3,4]
Padding
[1,2,3,4,0,0,0,0,0,0,0,]"""

In [ ]:
#Building the LSTM model
lstm_model = Sequential([Embedding(max_words,128,input_length=max_len),
                         LSTM(64),
                         Dropout(0.5),
                         Dense(1,activation='sigmoid')])
lstm_model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
lstm_model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#Train the LSTM model
lstm_history = lstm_model.fit(X_train_pad,y_train,epochs=10,batch_size=32,validation_split=0.2,verbose=1)

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 243ms/step - accuracy: 0.5141 - loss: 0.6949 - val_accuracy: 0.5188 - val_loss: 0.6930
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 75ms/step - accuracy: 0.5109 - loss: 0.6926 - val_accuracy: 0.5188 - val_loss: 0.6930
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step - accuracy: 0.4969 - loss: 0.6947 - val_accuracy: 0.4812 - val_loss: 0.6944
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 75ms/step - accuracy: 0.4984 - loss: 0.6965 - val_accuracy: 0.4812 - val_loss: 0.6933
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 77ms/step - accuracy: 0.4797 - loss: 0.6943 - val_accuracy: 0.4812 - val_loss: 0.6942
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 74ms/step - accuracy: 0.5281 - loss: 0.6935 - val_accuracy: 0.4812 - val_loss: 0.6936
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 84ms/step - accuracy: 0.4859 - loss: 0.6951 - val_accuracy: 0.5188 - val_loss: 0.6931
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 99ms/step - accuracy: 0.5109 - loss: 0.6964 - val_accuracy: 0.5188 - 

In [ ]:
lstm_prob = lstm_model.predict(X_test_pad,verbose=0)
lstm_pred = (lstm_prob >= 0.5).astype(int).flatten()
print("LSTM Accuracy:",accuracy_score(y_test,lstm_pred))
print(classification_report(y_test,lstm_pred,target_names=['Negative','Positive']))

LSTM Accuracy: 0.5
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00       100
    Positive       0.50      1.00      0.67       100

    accuracy                           0.50       200
   macro avg       0.25      0.50      0.33       200
weighted avg       0.25      0.50      0.33       200



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
print(confusion_matrix(y_test,lstm_pred))

[[  0 100]
 [  0 100]]


In [ ]:
#GRU model
"""gru_model = Sequential([Embedding(max_words,128,input_length=max_len),
                        GRU(64),
                        Dropout(0.5),
                        Dense(1,activation='sigmoid')])"""
gru_model = Sequential([
    Embedding(max_words,64,input_length=150),
    GRU(32,dropout=0.2),
    Dense(16,activation='relu'),
    Dropout(0.3),
    Dense(1,activation='sigmoid')
])
gru_model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
gru_model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
gru_history = gru_model.fit(X_train_pad,y_train,epochs=10,batch_size=32,validation_split=0.2,verbose=1)

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 70ms/step - accuracy: 0.4859 - loss: 0.6938 - val_accuracy: 0.4812 - val_loss: 0.6939
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.4891 - loss: 0.6936 - val_accuracy: 0.4812 - val_loss: 0.6933
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - accuracy: 0.5063 - loss: 0.6939 - val_accuracy: 0.4812 - val_loss: 0.6940
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.4844 - loss: 0.6947 - val_accuracy: 0.5188 - val_loss: 0.6931
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.4844 - loss: 0.6949 - val_accuracy: 0.4812 - val_loss: 0.6937
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.4797 - loss: 0.6933 - val_accuracy: 0.4812 - val_loss: 0.6932
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.4812 - loss: 0.6933 - val_accuracy: 0.4812 - val_loss: 0.6932
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 80ms/step - accuracy: 0.4953 - loss: 0.6935 - val_accuracy: 0.4812 - v

In [ ]:
gru_prob = gru_model.predict(X_test_pad,verbose=0)
gru_pred = (gru_prob >= 0.5).astype(int).flatten()
print("GRU Accuracy:",accuracy_score(y_test,gru_pred))
print(classification_report(y_test,gru_pred,target_names=['Negative','Positive']))
print(confusion_matrix(y_test,gru_pred))

GRU Accuracy: 0.5
              precision    recall  f1-score   support

    Negative       0.50      1.00      0.67       100
    Positive       0.00      0.00      0.00       100

    accuracy                           0.50       200
   macro avg       0.25      0.50      0.33       200
weighted avg       0.25      0.50      0.33       200

[[100   0]
 [100   0]]


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
lstm_accuracy = accuracy_score(y_test,lstm_pred)
gru_accuracy = accuracy_score(y_test,gru_pred)
results = pd.DataFrame({'Model':['LSTM','GRU'],'Accuracy':[lstm_accuracy,gru_accuracy]})
print(results)

  Model  Accuracy
0  LSTM       0.5
1   GRU       0.5
